In [104]:
%pwd

'A:\\Kidney-Health-Analysis-System'

In [ ]:
from dataclasses import dataclass
from pathlib import Path 

In [ ]:
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path 
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [107]:
from KidneyClassifier.constants import *
from KidneyClassifier.utils.common import read_yaml, create_directories

In [108]:
class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:

        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config


In [109]:
import os
import zipfile
import gdown

from pathlib import Path

from KidneyClassifier import logger
from KidneyClassifier.utils.common import get_size

In [110]:
class DataIngestion:

    def __init__(self, config):
        self.config = config

    def download_file(self) -> str:

        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file

            os.makedirs(self.config.root_dir, exist_ok=True)

            if not os.path.exists(zip_download_dir):

                logger.info(
                    f"Downloading data from {dataset_url} "
                    f"into file {zip_download_dir}"
                )

                file_id = dataset_url.split("/")[-2]

                prefix = "https://drive.google.com/uc?export=download&id="

                gdown.download(
                    prefix + file_id,
                    str(zip_download_dir),
                    quiet=False
                )

                logger.info(
                    f"Downloaded data from {dataset_url} "
                    f"into file {zip_download_dir}"
                )

            else:
                logger.info(
                    f"File already exists with size: "
                    f"{get_size(Path(zip_download_dir))}"
                )

        except Exception as e:
            raise e

    def extract_zip_file(self):

        unzip_path = self.config.unzip_dir

        os.makedirs(unzip_path, exist_ok=True)

        with zipfile.ZipFile(
            self.config.local_data_file, "r"
        ) as zip_ref:

            zip_ref.extractall(unzip_path)

        logger.info(
            f"Extracted ZIP file into directory: {unzip_path}"
        )


In [113]:
print(hasattr(DataIngestion, "extract_zip_file"))

True


In [114]:
try:

    config = ConfigurationManager()

    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(
        config=data_ingestion_config
    )

    data_ingestion.download_file()

    data_ingestion.extract_zip_file()

except Exception as e:
    raise e

[2026-11-26 15:30:08,987: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-11-26 15:30:09,083: INFO: common: yaml file: params.yaml loaded successfully]
[2026-11-26 15:30:09,092: INFO: common: created directory at: artifacts]
[2026-11-26 15:30:09,097: INFO: common: created directory at: artifacts/data_ingestion]
[2026-11-26 15:30:09,100: INFO: 83952922: Downloading data from https://drive.google.com/file/d/1zw63lweNjpzIBca1ygsOlHvyP81wzNfl/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?id=1zw63lweNjpzIBca1ygsOlHvyP81wzNfl
From (redirected): https://drive.google.com/uc?id=1zw63lweNjpzIBca1ygsOlHvyP81wzNfl&confirm=t&uuid=3bda92c9-5aed-477a-b29b-6ea70bae9f04
To: A:\Kidney-Health-Analysis-System\artifacts\data_ingestion\data.zip
100%|██████████| 940M/940M [05:15<00:00, 2.98MB/s] 


[2026-11-26 15:35:27,216: INFO: 83952922: Downloaded data from https://drive.google.com/file/d/1zw63lweNjpzIBca1ygsOlHvyP81wzNfl/view?usp=sharing into file artifacts/data_ingestion/data.zip]
[2026-11-26 15:36:08,907: INFO: 83952922: Extracted ZIP file into directory: artifacts/data_ingestion]
